# An M-Dwarf Flares Example

In this notebook we run a simulation of the built-in m-dwarf flares model from Mendoza et al. (2022), with free parameters for amplitude and full width half max of the flare.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.models.mdwarf_flare_model import MDwarfFlareModel
from lightcurvelynx.obstable.lsst_obstable import LSSTObsTable
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve, simulate_lightcurves
from lightcurvelynx.survey_info import SurveyInfo
from lightcurvelynx.utils.plotting import plot_lightcurves

## Fake Survey Data

For the purposes of this notebook, we create a very basic survey using the LSST instrument that looks at one region of the sky. It revisits the same patch `samples_per_day` times a night in alternating filters ("g", "r", "i", "z", and "u").

For more realistic surveys, see the examples for [Rubin OpSim](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/opsim_notebook.html), [Rubin CCDVisit Table](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/pre_executed/rubin_ccdvisit.html), or [Custom Surveys](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/custom_survey.html).

In [ ]:
filters = ["g", "r", "i", "z", "u"]
num_days = 4
samples_per_day = 100  # 20 samples in each filter
num_samples = num_days * samples_per_day

times = np.arange(num_samples) / samples_per_day + 60676.0  # MJD starting at 60676.0
fake_data = {
    "expMidptMJD": times,  # MJD
    "ra": np.full(num_samples, 150.0),  # Degrees
    "dec": np.full(num_samples, -30.0),  # Degrees
    "band": [filters[i % len(filters)] for i in range(num_samples)],
    "exptime": np.full(num_samples, 30.0),  # Seconds
    # Noise information is random from distributions around DP2 values.
    "pixelScale": np.full(num_samples, 0.2),  # Arcseconds per pixel
    "seeing": np.random.uniform(low=0.84, high=0.88, size=num_samples),  # arcseconds
    "skyBg": np.random.uniform(low=1420.0, high=1430.0, size=num_samples),  # ADU
    "zeroPoint": np.random.normal(
        loc=30.0, scale=0.5, size=num_samples
    ),  # magnitudes to produce 1 count (ADU)
}
obs_table = LSSTObsTable(fake_data)
obs_table.head()

Since we are using a standard survey type (LSST), we can just pass the name to the `SurveyInfo` info constructor and have it load the default passbands and noise model.

In [ ]:
survey_info = SurveyInfo(obstable=obs_table, survey_name="LSST")

## Create Parameter Distributions

With LightCurveLynx model parameters may be both fixed numerical values or to be drawn from random distributions. Here we create samplers for the various m-dwarf flare parameters, including: star_temp, star_radius, flare_fwhm, and flare_amplitude. 

The m-dwarf flares model has a built-in prior model for these parameters if they are not specified.

In [ ]:
# Star temperature is fixed at 4000 K.
star_temp = 4000

# Star radius is drawn from a normal distribution centered at 0.09 solar radii.
star_radius_sampler = NumpyRandomFunc(
    "normal",
    loc=0.09 * u.R_sun.to(u.cm),
    scale=0.01 * u.R_sun.to(u.cm),
)

# Flare FWHM is drawn from a uniform distribution [0.05, 0.15] in the same units as t0.
flare_fwhm_sampler = NumpyRandomFunc("uniform", low=0.05, high=0.15)

# Flare amplitude is drawn from a normal distribution centered at 0.05.
flare_amplitude_sampler = NumpyRandomFunc("normal", loc=0.05, scale=0.01)

# t0 is fixed at the midpoint of our fake OpSim coverage, so we can see the flare clearly.
t0 = (times.min() + times.max()) / 2

## Create the Model

Now that we have the parameter setters, we can define the m-dwarf flare mode.

In [ ]:
model = MDwarfFlareModel(
    star_temp=star_temp,
    star_radius=star_radius_sampler,
    flare_fwhm=flare_fwhm_sampler,
    flare_amplitude=flare_amplitude_sampler,
    t0=t0,
    ra=150.0,  # We always see the object
    dec=-30.0,  # We always see the object
    distance=100.0,  # Fixed distance in parsecs
    node_label="model",
)

We can generate samples to see the values of the parameters.

In [ ]:
state = model.sample_parameters(num_samples=10)
for key, value in state["model"].items():
    print(f"{key}: {value}")

## Run Simulations

Now let's simulate a few light curves!

In [ ]:
# Fix the random seed to make it reproducible.
rng = np.random.default_rng(42)

df = simulate_lightcurves(
    model=model,
    num_samples=3,
    survey_info=survey_info,
    rng=rng,
)

# params is too large to show =)
params = df["params"].copy()
df = df.drop(columns=["params"])

## Plot few light curves

We plot a few of the simulated light curves in magnitude space.

In [ ]:
for idx in range(3):
    # Extract the row for this object.
    row = df.iloc[idx]
    lc = row["lightcurve"]

    # Extract the parameters used during the simulation of this object. Use that to
    # compute the noise-free light curve for this object, which we will plot alongside
    # the simulated light curve.
    noise_free_lcs = compute_single_noise_free_lightcurve(
        model,
        GraphState.from_dict(params[idx]),
        survey_info.passbands,
        rest_frame_phase_min=-num_days / 2,
        rest_frame_phase_max=num_days / 2,
        rest_frame_phase_step=0.01,
    )

    plot_lightcurves(
        fluxes=lc["flux"],
        times=lc["mjd"],
        fluxerrs=lc["fluxerr"],
        filters=lc["filter"],
        underlying_model=noise_free_lcs,
        plot_magnitudes=True,
    )

    # Plot a vertical line at t0, which is the reference time for the light curve.
    plt.axvline(x=params[idx]["model.t0"], color="k", linestyle="--")
    plt.show()